# Few-Shot and Chain-of-Thought

**Phase 01** · ~60 minutes · Python

**Prerequisites:** See lesson README

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thepandanlabs/applied-ai-from-scratch/blob/main/notebooks/phase-01/03-few-shot-and-chain-of-thought.ipynb)

---

> This notebook is auto-generated from the [Applied AI From Scratch](https://github.com/thepandanlabs/applied-ai-from-scratch) curriculum.  
> Run cells top to bottom. Set your API key in the **Setup** cell below.


In [ ]:
!pip install -q anthropic

import os

try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    pass  # Running locally — set ANTHROPIC_API_KEY before this cell

print("Setup complete. API key set:", bool(os.environ.get("ANTHROPIC_API_KEY")))

## Implementation

Run each cell in order:

### Setup & Imports

In [ ]:
"""
Lesson 01-03: Few-Shot and Chain-of-Thought
Phase 01: Prompt and Context Engineering

Three experiments comparing zero-shot, few-shot, and CoT prompting
on a severity classification task. Includes the SDK messages-array
pattern for structured few-shot examples.
"""

import anthropic

client = anthropic.Anthropic()

### Test tickets covering the full severity range

In [ ]:
TEST_TICKETS = [
    # Clear Critical
    "The entire payment service is down. No transactions are going through for any users.",
    # Clear High
    "Our export to CSV feature is producing incorrect totals. Users cannot reconcile their data.",
    # Ambiguous Medium/High
    "The search results sometimes show duplicates when filtering by date range.",
    # Clear Low
    "The font in the mobile app looks slightly different than the web version.",
    # Ambiguous Critical/High
    "Users in the EU region cannot log in. This has been broken for 30 minutes.",
    # Clear Low
    "How do I change my billing address?",
    # Ambiguous High/Critical
    "Our webhook delivery rate dropped from 99.9% to 60% starting an hour ago.",
]

# Ground truth labels for evaluation (set manually)
GROUND_TRUTH = [
    "Critical",
    "High",
    "Medium",
    "Low",
    "Critical",
    "Low",
    "High",
]

### Experiment 1: Zero-shot baseline

In [ ]:
ZERO_SHOT_PROMPT = """Classify the severity of this customer support ticket.
Severity levels: Critical, High, Medium, Low.

Definitions:
- Critical: complete outage, data loss, affects all/most users, revenue impact
- High: major feature broken, significant user impact, no workaround
- Medium: partial impact, workaround exists, affects some users
- Low: cosmetic issue, question, feature request, minor inconvenience

Output only the severity level. No explanation.

Ticket: {ticket}"""


def classify_zero_shot(ticket: str) -> str:
    response = client.messages.create(
        model="claude-3-5-haiku-20241022",
        max_tokens=16,
        messages=[{
            "role": "user",
            "content": ZERO_SHOT_PROMPT.format(ticket=ticket)
        }]
    )
    return response.content[0].text.strip()

### Experiment 2: Few-shot (inline examples in prompt)

In [ ]:
FEW_SHOT_PROMPT = """Classify the severity of this customer support ticket.
Severity levels: Critical, High, Medium, Low.

Examples:
Ticket: Our API is returning 500 errors for all requests. Production is down.
Severity: Critical

Ticket: The export to CSV function is producing incorrect totals.
Severity: High

Ticket: The search results sometimes show duplicates.
Severity: Medium

Ticket: Can you add a dark mode option to the app?
Severity: Low

Ticket: Half our users in Asia-Pacific cannot authenticate. Ongoing for 1 hour.
Severity: Critical

Ticket: The mobile notification sound plays twice occasionally.
Severity: Low

Output only the severity level. No explanation.

Ticket: {ticket}"""


def classify_few_shot(ticket: str) -> str:
    response = client.messages.create(
        model="claude-3-5-haiku-20241022",
        max_tokens=16,
        messages=[{
            "role": "user",
            "content": FEW_SHOT_PROMPT.format(ticket=ticket)
        }]
    )
    return response.content[0].text.strip()

### Experiment 3: Chain-of-thought (reasoning before answer)

In [ ]:
COT_SYSTEM = """You are a support ticket triage specialist.

When classifying tickets, reason through these questions:
1. Who is affected: one user, some users, or all/most users?
2. Is core functionality or revenue directly impacted?
3. Is this time-sensitive with no workaround available?

Severity definitions:
- Critical: complete outage, data loss, broad user impact, no workaround
- High: major feature broken, significant impact, limited workaround
- Medium: partial impact, workaround available, affects some users
- Low: cosmetic, question, feature request, minor inconvenience"""

COT_USER_TEMPLATE = """Reason through the 3 triage questions for this ticket,
then output your final answer on a line by itself as: SEVERITY: [level]

Ticket: {ticket}"""


def classify_cot(ticket: str) -> tuple[str, str]:
    """Returns (severity_label, full_reasoning)."""
    response = client.messages.create(
        model="claude-3-5-haiku-20241022",
        max_tokens=512,
        system=COT_SYSTEM,
        messages=[{
            "role": "user",
            "content": COT_USER_TEMPLATE.format(ticket=ticket)
        }]
    )
    full_text = response.content[0].text.strip()

    # Extract the severity label from the last line
    label = "Unknown"
    for line in full_text.split("\n"):
        if line.strip().startswith("SEVERITY:"):
            label = line.split(":", 1)[-1].strip()
            break

    return label, full_text

### Experiment 4: Few-shot using SDK messages array (structured, no inline examples)

In [ ]:
FEW_SHOT_EXAMPLES = [
    ("Our API is returning 500 errors for all requests. Production is down.", "Critical"),
    ("The export to CSV function is producing incorrect totals.", "High"),
    ("The search results sometimes show duplicates.", "Medium"),
    ("Can you add a dark mode option to the app?", "Low"),
    ("Half our users in Asia-Pacific cannot authenticate. Ongoing for 1 hour.", "Critical"),
    ("The mobile notification sound plays twice occasionally.", "Low"),
]


def classify_few_shot_sdk(ticket: str) -> str:
    """
    Few-shot using the messages array pattern.
    Examples are user/assistant turn pairs, keeping them separate from instructions.
    """
    messages = []

    # Add each example as a user/assistant pair
    for example_ticket, example_label in FEW_SHOT_EXAMPLES:
        messages.append({"role": "user",      "content": f"Ticket: {example_ticket}"})
        messages.append({"role": "assistant", "content": example_label})

    # Add the real ticket
    messages.append({"role": "user", "content": f"Ticket: {ticket}"})

    response = client.messages.create(
        model="claude-3-5-haiku-20241022",
        max_tokens=16,
        system=(
            "You are a support ticket triage specialist. "
            "Classify severity: Critical, High, Medium, Low. "
            "Output only the label."
        ),
        messages=messages
    )
    return response.content[0].text.strip()

### Evaluation: compare all approaches against ground truth

In [ ]:
def evaluate_all() -> None:
    print("=" * 70)
    print("EVALUATION: Zero-shot vs Few-shot vs CoT vs Few-shot (SDK)")
    print("=" * 70)

    results = {
        "zero_shot":      [],
        "few_shot":       [],
        "cot":            [],
        "few_shot_sdk":   [],
    }

    for i, ticket in enumerate(TEST_TICKETS):
        expected = GROUND_TRUTH[i]
        print(f"\nTicket {i+1}: {ticket[:60]}...")
        print(f"  Expected: {expected}")

        zs  = classify_zero_shot(ticket)
        fs  = classify_few_shot(ticket)
        cot_label, _ = classify_cot(ticket)
        fs_sdk = classify_few_shot_sdk(ticket)

        results["zero_shot"].append(zs == expected)
        results["few_shot"].append(fs == expected)
        results["cot"].append(cot_label == expected)
        results["few_shot_sdk"].append(fs_sdk == expected)

        print(f"  Zero-shot:    {zs:10s}  {'OK' if zs == expected else 'WRONG'}")
        print(f"  Few-shot:     {fs:10s}  {'OK' if fs == expected else 'WRONG'}")
        print(f"  CoT:          {cot_label:10s}  {'OK' if cot_label == expected else 'WRONG'}")
        print(f"  Few-shot SDK: {fs_sdk:10s}  {'OK' if fs_sdk == expected else 'WRONG'}")

    print("\n" + "=" * 70)
    print("SUMMARY")
    print("=" * 70)
    n = len(TEST_TICKETS)
    for method, correct_list in results.items():
        score = sum(correct_list)
        print(f"  {method:20s}: {score}/{n} ({100*score//n}%)")

### Demo: Show CoT reasoning for one ticket

In [ ]:
def demo_cot_reasoning() -> None:
    print("\n" + "=" * 70)
    print("DEMO: CoT reasoning trace for an ambiguous ticket")
    print("=" * 70)

    ambiguous = "Our webhook delivery rate dropped from 99.9% to 60% starting an hour ago."
    label, reasoning = classify_cot(ambiguous)

    print(f"Ticket: {ambiguous}")
    print(f"\nFull CoT response:\n{reasoning}")
    print(f"\nExtracted severity: {label}")


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------

### Demo

In [ ]:
print("Lesson 01-03: Few-Shot and Chain-of-Thought")
print("\nOptions:")
print("  1. Run full evaluation (all 7 tickets, all 4 methods)")
print("  2. Demo CoT reasoning trace")
print("  3. Run both\n")

choice = input("Choice [1/2/3]: ").strip()

if choice == "1":
    evaluate_all()
elif choice == "2":
    demo_cot_reasoning()
else:
    demo_cot_reasoning()
    evaluate_all()

---

## Self-Check

Answer these without running code first:

**1. What does the variance pattern tell you, and what should you try next?**

- A. The variance is too high; switch to a deterministic model.
- B. Few-shot has reached its ceiling for this task. The remaining errors are likely on ambiguous edge cases that need reasoning. Apply CoT for those cases.
- C. Add more examples until you reach 12-15 total.
- D. The variance is normal; 88% is the realistic ceiling for this task.

**2. What does this result tell you about CoT and this specific task?**

- A. CoT is broken for extraction tasks.
- B. The model needs more examples before CoT will work.
- C. Product name extraction does not require multi-step reasoning. CoT adds cost with no accuracy benefit. Remove it and use few-shot or zero-shot.
- D. CoT output tokens should not increase this much; there is a prompt error.

**3. Why did adding more examples hurt performance, and what is the fix?**

- A. 20 examples exceed the context window.
- B. Too many examples dilute the pattern the model is trying to learn. The model spends more attention on example diversity than on the pattern. Use 3-6 well-chosen examples per label.
- C. The 20 examples include incorrect labels that are misleading the model.
- D. Few-shot performance always decreases after 10 examples due to attention limits.

**4. What is the practical production advantage of the messages array approach?**

- A. The messages array is processed faster by the API.
- B. Examples stored separately from prompt logic can be updated without changing code. You can A/B test example sets, add domain-specific examples per customer, or reload examples without a deployment.
- C. The messages array supports more examples than inline prompt strings.
- D. Inline examples cause the model to ignore the system prompt.

**5. The CoT reasoning reached the wrong conclusion. What likely went wrong?**

- A. CoT cannot be used for classification tasks.
- B. The reasoning chain weighted 'payment is core' too heavily relative to 'regional scope'. The CoT instruction did not specify how to weight competing factors. Update the system prompt to add explicit weighting rules.
- C. The model should have been given more examples of High tickets before applying CoT.
- D. The correct answer is Critical, not High. The ground truth label is wrong.

**6. How do you get the best of both without the regression on clear cases?**

- A. Remove the CoT reasoning from the examples so they are labels only.
- B. Use a routing layer: fast few-shot (no CoT) for high-confidence cases, few-shot+CoT for ambiguous ones.
- C. Switch to a larger model that handles both without regression.
- D. Accept the trade-off. 95% on clear cases is still good.

**7. **

- A. Pick the 6 most recent examples to ensure they reflect current document formats.
- B. Pick 6 random examples to get unbiased coverage.
- C. Pick examples that cover the full output distribution, include at least one edge case, and are representative of the actual input distribution you'll encounter in production.
- D. Pick the 6 examples where the extraction was most straightforward to avoid confusing the model.

**8. What is the correct mechanistic explanation for why CoT improves reasoning on multi-step tasks?**

- A. CoT increases the model's temperature, which produces more diverse and creative outputs.
- B. CoT allocates more memory to the problem by storing intermediate steps.
- C. CoT forces the model to generate intermediate reasoning tokens before the answer token. Each token becomes part of the context the model uses when predicting the next token, giving it more information at the point of committing to the final answer.
- D. CoT works by activating a different neural pathway in the model that is better suited to reasoning.

_Answers are in `checks.json` in the lesson directory._